***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json 
pd.set_option('display.max_columns', None)


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'EPA')

path_code    = os.path.join(path_git, 'Data', 'EPA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Importing

***

In [ ]:

print('AQI data:')
df_aqi = pd.read_excel(os.path.join(path_out, 'Health_3_MSA_EPA_raw.xlsx'))
print(df_aqi.shape)
display(df_aqi.head(3))
print('');print('')

df_aqi = df_aqi[~df_aqi['aqi'].isna()]
df_aqi = df_aqi[~df_aqi['pollutant_standard'].isna()]
# df_aqi = df_aqi[df_aqi['sample_duration'].isin(['24-HR BLK AVG', '24 HOUR'])]
# df_aqi = df_aqi[df_aqi['pollutant_standard'].str.contains('2015')]


aqi_grouping = ['cbsa_code', 'cbsa', 'date_local', 'Year_Imported']

df_aqi = df_aqi.groupby(aqi_grouping, as_index=False)['aqi'].max()

df_aqi['aqi'] = round(df_aqi['aqi'])

df_aqi['violation'] = 0
df_aqi.loc[df_aqi['aqi'] > 100, 'violation'] = 1
# df_aqi.loc[(df_aqi['aqi'] >= 101) & (df_aqi['aqi'] <= 150), 'violation'] = 1


df_aqi = df_aqi[df_aqi['violation'] > 0]
df_aqi = df_aqi.reset_index(drop=True)

print(df_aqi.shape)
display(df_aqi.head())

print(df_aqi[(df_aqi['Year_Imported'] == 2021) & (df_aqi['cbsa_code'] == 40900)].shape)

In [ ]:
print('AQI data:')
df_aqi = pd.read_excel(os.path.join(path_out, 'Health_3_MSA_EPA_raw.xlsx'))
print(df_aqi.shape)
print(df_aqi['sample_duration'].unique())
print(df_aqi['pollutant_standard'].unique())
print('');print('')

In [ ]:
df_aqi = df_aqi[df_aqi['parameter'] == 'Ozone']
# df_aqi = df_aqi[df_aqi['sample_duration'] != '1 HOUR']
df_aqi = df_aqi[df_aqi['pollutant_standard'] == 'Ozone 8-hour 2015']
df_aqi = df_aqi.sort_values(['site_number', 'date_local'])
df_aqi = df_aqi[(df_aqi['Year_Imported'] == 2021) & (df_aqi['cbsa_code'] == 40900)]

df_aqi = df_aqi[~df_aqi['aqi'].isna()]

# threshold = 100

# aqi_grouping = ['cbsa_code', 'cbsa', 'date_local', 'parameter', 'Year_Imported']

# df_aqi = df_aqi.groupby(aqi_grouping, as_index=False)['aqi'].mean()

# df_aqi['aqi'] = round(df_aqi['aqi'])

# df_aqi['violation'] = 0
# df_aqi.loc[df_aqi['aqi'] >= threshold, 'violation'] = 1

# aqi_grouping = ['cbsa_code', 'cbsa', 'date_local', 'Year_Imported']

# df_aqi = df_aqi.groupby(aqi_grouping, as_index=False)['violation'].mean()

# df_aqi = df_aqi[df_aqi['violation'] > 0]
# df_aqi = df_aqi.reset_index(drop=True)

df_aqi.head()

In [ ]:
df_aqi.shape

In [ ]:
df_qc = df_aqi[df_aqi['aqi'] > 100]
df_qc.head()

In [ ]:
df_aqi[(df_aqi['date_local'] == '2021-05-13')]

In [ ]:
px.histogram(df_aqi, x = 'aqi')

In [ ]:
# print('PM25 data:')
# df_pm25 = pd.read_excel(os.path.join(path_out, 'Health_3_MSA_EPA_PM25_raw.xlsx'))
# print(df_pm25.shape)
# display(df_pm25.head(3))
# print('');print('')

# print('O3 data:')
# df_O3 = pd.read_excel(os.path.join(path_out, 'Health_3_MSA_EPA_O3_raw.xlsx'))
# print(df_O3.shape)
# display(df_O3.head(3))
# print('')

# df_pm25 = df_pm25[~df_pm25['aqi'].isna()]
# df_O3   = df_O3  [~df_O3  ['aqi'].isna()]

# threshold = 100

# aqi_grouping = ['cbsa_code', 'cbsa', 'county', 'state_code', 'county_code', 'site_number', 'date_local', 'local_site_name', 'parameter']

# df_pm25 = df_pm25.groupby(aqi_grouping, as_index=False)['aqi'].mean()
# df_O3   = df_O3  .groupby(aqi_grouping, as_index=False)['aqi'].mean()

# df_pm25['aqi'] = round(df_pm25['aqi'])
# df_O3  ['aqi'] = round(df_O3  ['aqi'])

# df_aqi = pd.concat([df_pm25, df_O3])

# df_aqi['violation'] = 0
# df_aqi.loc[df_aqi['aqi'] >= threshold, 'violation'] = 1

# aqi_grouping = ['cbsa_code', 'cbsa', 'county', 'state_code', 'county_code', 'site_number', 'date_local', 'local_site_name']

# df_aqi = df_aqi.groupby(aqi_grouping, as_index=False)['violation'].mean()

# df_aqi = df_aqi[df_aqi['violation'] > 0]
# df_aqi = df_aqi.reset_index(drop=True)

# print(df_aqi.shape)
# display(df_aqi.head())